In [1]:
import pandas as pd
import requests
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import sys

In [2]:
CSV_PATH = '../data/preprocessed_shorts.csv'
OUTPUT_DIR = '../thumbnails' 
MAX_WORKERS = 10

In [3]:
def download_thumbnail(video_id):
    urls_to_try = [
        f"https://img.youtube.com/vi/{video_id}/maxresdefault.jpg",
        f"https://img.youtube.com/vi/{video_id}/hqdefault.jpg"    
    ]
    
    filepath = os.path.join(OUTPUT_DIR, f"{video_id}.jpg")
    
    if os.path.exists(filepath):
        return f"[PULADO] {video_id}: Imagem já existe."

    for url in urls_to_try:
        try:
            response = requests.get(url, timeout=10)
    
            if response.status_code == 200:
                with open(filepath, 'wb') as file:
                    file.write(response.content)
                resolucao = "MaxRes" if "maxres" in url else "HQ"
                return f"[SUCESSO] {video_id}: Baixado em {resolucao}."
                
        except requests.exceptions.RequestException as e:
            return f"[ERRO DE CONEXÃO] {video_id}: {e}"
            
    return f"[FALHA] {video_id}: Nenhuma thumbnail encontrada."

In [4]:
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
    print(f"Diretório '{OUTPUT_DIR}' criado.")

# Carrega o CSV e limpa os IDs
try:
    df = pd.read_csv(CSV_PATH)
except FileNotFoundError:
    print(f"Erro: O arquivo '{CSV_PATH}' não foi encontrado.")
    sys.exit(1)

if 'videoId' not in df.columns:
    print("Erro: A coluna 'videoId' não foi encontrada no seu CSV.")
    sys.exit(1)

# Remove valores nulos e garante que sejam strings únicas
video_ids = df['videoId'].dropna().astype(str).unique()
total_videos = len(video_ids)

print(f"Iniciando o download de {total_videos} thumbnails...")

# download em paralelo
sucessos = 0
falhas = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(download_thumbnail, vid): vid for vid in video_ids}
    
    for future in as_completed(futures):
        resultado = future.result()
        print(resultado)
        
        if "[SUCESSO]" in resultado or "[PULADO]" in resultado:
            sucessos += 1
        else:
            falhas += 1

print("RESUMO DO DOWNLOAD")
print(f"Total processado: {total_videos}")
print(f"Sucessos/Pulados: {sucessos}")
print(f"Falhas:           {falhas}")

Diretório '../thumbnails' criado.
Iniciando o download de 9270 thumbnails...
[SUCESSO] -7n-QWop-4M: Baixado em MaxRes.
[SUCESSO] ZzrG8BgHoZ0: Baixado em MaxRes.
[SUCESSO] wpRkmL6Hbbo: Baixado em MaxRes.
[SUCESSO] w22lrhGXy_I: Baixado em MaxRes.
[SUCESSO] 0kiWV3FJik0: Baixado em MaxRes.
[SUCESSO] pmmwmIJZ5_k: Baixado em MaxRes.
[SUCESSO] glr13drvkhE: Baixado em MaxRes.
[SUCESSO] GetFGAYgZGs: Baixado em MaxRes.
[SUCESSO] Eo25bych-sI: Baixado em MaxRes.
[SUCESSO] diiNz7u3QEs: Baixado em MaxRes.
[SUCESSO] AKpqzVycj6s: Baixado em MaxRes.
[SUCESSO] xpujlmGrgk0: Baixado em MaxRes.
[SUCESSO] xto8YIDTuIg: Baixado em MaxRes.
[SUCESSO] JRz8c_hCthc: Baixado em MaxRes.
[SUCESSO] WPDatqSRIvc: Baixado em MaxRes.
[SUCESSO] YdzmmRWERWI: Baixado em MaxRes.
[SUCESSO] uB5VINOvdLQ: Baixado em MaxRes.
[SUCESSO] mzgr9E3zMg0: Baixado em MaxRes.
[SUCESSO] 8G60ahci4is: Baixado em HQ.
[SUCESSO] NAjhZCyUKsM: Baixado em MaxRes.
[SUCESSO] leJw7LAiVHk: Baixado em MaxRes.
[SUCESSO] pVScL5-JMbM: Baixado em MaxRes.
[SU